# Interview Prep: 02 - Advanced Internals

A senior Python role requires deep knowledge of how the language works under the hood. This notebook covers **Memory Management**, the **GIL (and its removal)**, and advanced features like **Metaclasses** and **Descriptors**.

---

## 1. Memory Management & GC

### **Interview Question**: "How does Python's Garbage Collection work? Why is reference counting not enough?"

#### **Key Talking Points**:
- **Reference Counting**: Primary mechanism. When `sys.getrefcount(obj)` hits 0, it's deleted immediately.
- **Generational GC**: Handles **Cyclic References** (e.g., A refers to B, and B refers to A, so their ref counts never hit 0). 
- **Generations**: Python divides objects into 3 generations (0, 1, 2) based on how many walk-throughs they survive.

---

## 2. The GIL & PEP 703 (No-GIL)

### **Interview Question**: "What is the GIL, and what is the current state of it in Python 3.13?"

#### **Key Talking Points**:
- **GIL (Global Interpreter Lock)**: A mutex that protects access to Python objects, preventing multiple threads from executing Python bytecodes at once.
- **Impact**: Makes multi-threading useless for CPU-bound tasks in standard CPython.
- **PEP 703**: Introduced "Free-threaded Python" (experimental in 3.13). It allows running Python without the GIL by using **Biased Reference Counting** and per-object locks.

---

## 3. Metaclasses and `__new__` vs `__init__`

### **Interview Question**: "Explain the purpose of a Metaclass. When would you actually use one?"

**Answer**: A metaclass is the "class of a class." It allows you to intercept the creation of a class. 
- **Usage**: Custom validation at class creation time, automatically registering classes, or implementing Singletons.
- **Difference**: `__new__` is called *before* `__init__`. It's the method that actually *creates* the object instance.

---

## 4. Descriptors: Behind the Scenes of Properties

### **Interview Question**: "How does the `@property` decorator actually work?"

**The Logic**: `@property` is built using the **Descriptor Protocol** (`__get__`, `__set__`, `__delete__`). Any object that defines these methods can customize attribute access.

#### **Challenge**: Implement a Descriptor that ensures a value is always positive.


In [ ]:
class PositiveValue:
    def __init__(self, name):
        self.name = name

    def __get__(self, instance, owner):
        return instance.__dict__.get(self.name)

    def __set__(self, instance, value):
        if value < 0:
            raise ValueError(f"{self.name} must be positive!")
        instance.__dict__[self.name] = value

class Product:
    price = PositiveValue("price")
    
    def __init__(self, name, price):
        self.name = name
        self.price = price

try:
    p = Product("Laptop", 1000)
    print(f"Product: {p.name}, Price: {p.price}")
    p.price = -10 # This should fail
except ValueError as e:
    print(f"Error caught: {e}")

## 5. Memory Profiling

### **Interview Question**: "How do you find a memory leak in a large Python application?"

**Strategy**:
1. **tracemalloc**: Built-in module to take snapshots of memory allocations.
2. **objgraph**: Great for visualizing object life-cycles and finding cyclic references.
3. **gc.get_objects()**: Inspecting the collector directly (last resort).

---